# PBL Fase 6 – Visão Computacional com YOLO
**Aluno:** Patrick Borges Melo · RM 574030 · **Classes:** `vaca` (0) e `caminhao` (1)

> Esqueleto gerado para a parte do dataset/treino. Ajuste os títulos e entregáveis conforme as issues do repositório.

## 1. Ambiente (rodar no Google Colab, com GPU: *Ambiente de execução → Alterar tipo → GPU*)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/ultralytics/yolov5 /content/yolov5
%cd /content/yolov5
!pip install -q -r requirements.txt
import torch; print(torch.__version__, 'GPU:', torch.cuda.is_available())

## 2. Dataset (Issues #1 e #2)
O dataset fica no Drive em *Espaço Matopiba / Fase 6 - Cap 1 / dataset*. Copiamos para o disco do Colab (mais rápido e evita problemas com espaços e acentos no caminho).

In [ ]:
import shutil, os
DRIVE_DATASET = '/content/drive/MyDrive/Espaço Matopiba/Fase 6 - Cap 1/dataset'
LOCAL = '/content/dataset'
if os.path.exists(LOCAL): shutil.rmtree(LOCAL)
shutil.copytree(DRIVE_DATASET, LOCAL)

# data.yaml com caminho do Colab (sobrescreve o do Drive)
open(f'{LOCAL}/data.yaml','w').write('''path: /content/dataset
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: vaca
  1: caminhao
''')
for s in ('train','val','test'):
    im = len(os.listdir(f'{LOCAL}/images/{s}')); lb = len(os.listdir(f'{LOCAL}/labels/{s}'))
    print(s, 'imagens', im, 'labels', lb)

Verificação rápida: cada imagem deve ter seu `.txt`, e as contagens esperadas são **64 / 8 / 8** (train/val/test).
Total de caixas: **129 vacas e 44 caminhões**.

In [ ]:
import glob, random, cv2
from google.colab.patches import cv2_imshow
def mostrar(split='train', n=3):
    for p in random.sample(glob.glob(f'{LOCAL}/images/{split}/*.jpg'), n):
        im = cv2.imread(p); h, w = im.shape[:2]
        lab = p.replace('images','labels').replace('.jpg','.txt')
        for l in open(lab):
            c,cx,cy,bw,bh = l.split(); cx,cy,bw,bh = float(cx)*w,float(cy)*h,float(bw)*w,float(bh)*h
            cv2.rectangle(im,(int(cx-bw/2),int(cy-bh/2)),(int(cx+bw/2),int(cy+bh/2)),(0,0,255) if c=='0' else (255,120,0),3)
        cv2_imshow(cv2.resize(im,(w//2,h//2)))
mostrar()

## 3. YOLO tradicional (pré-treinado no COCO, sem treinar)
O COCO já tem `cow` (19) e `truck` (7). Servirá de comparação com o modelo treinado por nós.

In [ ]:
!python detect.py --weights yolov5s.pt --source /content/dataset/images/test --classes 19 7 --conf 0.25 --name baseline_coco --exist-ok

## 4. YOLO customizado (treino)

In [ ]:
!python train.py --img 640 --batch 16 --epochs 100 --data /content/dataset/data.yaml --weights yolov5s.pt --patience 30 --name vaca_caminhao --exist-ok

> Os resultados do treino ficam em `runs/train/vaca_caminhao/` (não em `runs/detect/`).

## 5. Métricas

In [ ]:
import pandas as pd
r = pd.read_csv('/content/yolov5/runs/train/vaca_caminhao/results.csv'); r.columns = [c.strip() for c in r.columns]
r[['epoch','metrics/precision','metrics/recall','metrics/mAP_0.5','metrics/mAP_0.5:0.95']].tail()

In [ ]:
from IPython.display import Image
Image('/content/yolov5/runs/train/vaca_caminhao/results.png')

In [ ]:
Image('/content/yolov5/runs/train/vaca_caminhao/confusion_matrix.png')

### Avaliação no conjunto de teste (imagens que o modelo nunca viu)

In [ ]:
!python val.py --weights runs/train/vaca_caminhao/weights/best.pt --data /content/dataset/data.yaml --task test --img 640 --name teste_final --exist-ok

## 6. Inferência com o modelo treinado

In [ ]:
!python detect.py --weights runs/train/vaca_caminhao/weights/best.pt --source /content/dataset/images/test --conf 0.25 --name custom_test --exist-ok

In [ ]:
import glob
for p in sorted(glob.glob('/content/yolov5/runs/detect/custom_test/*.jpg'))[:4]:
    display(Image(p))

## 7. Comparação e conclusões
*Preencher:* YOLO tradicional (COCO) × YOLO customizado — acertos, falsos positivos, mAP, limitações do dataset (80 imagens, picapes contadas como caminhão, rebanhos densos).

In [ ]:
# Salvar resultados no Drive
dest = '/content/drive/MyDrive/Espaço Matopiba/Fase 6 - Cap 1/resultados_patrick'
shutil.copytree('/content/yolov5/runs', dest, dirs_exist_ok=True); print('salvo em', dest)